### Import the libraries required for data loading, cleaning, analysis, visualization,
### and statistical modeling.

In [2]:
import matplotlib
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import statsmodels

In [3]:
df_raw=pd.read_csv("../data/raw/ecommerce_sales_data.csv",dtype=str,keep_default_na=False)

In [4]:
df = df_raw.copy(deep=True)

In [5]:
df.head()

,Order ID,Customer_ID,Order Date,Category,Product Name,Qty,Unit Price,Country,PaymentMethod,customer_email,Rating,Discount,Returned?
0,ORD1031,CUST114,2023/04/22,Clothing,Shampoo,1,,Australia,Credit Card,customer27@mail.com,2,,No
1,ORD1109,CUST33,30.06.2023,Beauty,,10,free,France,PayPal,customer80@mail.com,5,0.41,
2,ORD1136,CUST5,2024/02/16,beauty,T-Shirt,8,28.03,INDIA,paypal,customer111@mail.com,1,0.1,Yes
3,ORD1088,CUST188,19-Sep-2024,Clothing,T-Shirt,9,111.45,Australia,Credit Card,customer166@mail.com,4,13%,Yes
4,ORD1920,CUST75,2024-04-21,beauty,Bluetooth Speaker,1,199.25,Germany,paypal,customer63@mail.com,2,16%,Yes


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Order ID        1031 non-null   object
 1   Customer_ID     1031 non-null   object
 2     Order Date    1031 non-null   object
 3   Category        1031 non-null   object
 4   Product Name    1031 non-null   object
 5   Qty             1031 non-null   object
 6   Unit Price      1031 non-null   object
 7   Country         1031 non-null   object
 8   PaymentMethod   1031 non-null   object
 9   customer_email  1031 non-null   object
 10  Rating          1031 non-null   object
 11  Discount        1031 non-null   object
 12  Returned?       1031 non-null   object
dtypes: object(13)
memory usage: 104.8+ KB


In [7]:
df.shape

(1031, 13)

In [8]:
df.columns = df.columns.str.strip()

In [9]:
empty_rows = ((df == "") | (df.isna())).all(axis=1)
df.index[empty_rows].tolist()

[159]

In [10]:
disguised_markers = ['', 'na', 'n/a', 'unknown', 'invalid_date', 'nan']
list_of_all_column=[column_name for column_name in df.columns]

In [11]:
missing_summary = []
for correct_column in list_of_all_column:
     null_count = df[correct_column].isna().sum()
     cleaned_values = (
        df[correct_column]
        .astype("string")
        .str.strip()
        .str.lower()
     )
     disguised_count = cleaned_values.isin(disguised_markers).sum()
     total_missing = null_count + disguised_count
     missing_percentage=(total_missing/len(df))*100
     missing_summary.append({
        "Column": correct_column,
        "Total Missing": total_missing,
        "Missing Percentage": missing_percentage
     })


In [12]:
missing_summary

[{'Column': 'Order ID',
  'Total Missing': np.int64(1),
  'Missing Percentage': np.float64(0.09699321047526674)},
 {'Column': 'Customer_ID',
  'Total Missing': np.int64(34),
  'Missing Percentage': np.float64(3.2977691561590685)},
 {'Column': 'Order Date',
  'Total Missing': np.int64(34),
  'Missing Percentage': np.float64(3.2977691561590685)},
 {'Column': 'Category',
  'Total Missing': np.int64(1),
  'Missing Percentage': np.float64(0.09699321047526674)},
 {'Column': 'Product Name',
  'Total Missing': np.int64(93),
  'Missing Percentage': np.float64(9.020368574199807)},
 {'Column': 'Qty',
  'Total Missing': np.int64(39),
  'Missing Percentage': np.float64(3.7827352085354025)},
 {'Column': 'Unit Price',
  'Total Missing': np.int64(26),
  'Missing Percentage': np.float64(2.521823472356935)},
 {'Column': 'Country',
  'Total Missing': np.int64(194),
  'Missing Percentage': np.float64(18.816682832201746)},
 {'Column': 'PaymentMethod',
  'Total Missing': np.int64(233),
  'Missing Percentage

In [13]:
duplicate_mask = df.duplicated(keep=False)
duplicate_count = duplicate_mask.sum()
duplicate_count

np.int64(36)

In [14]:
order_id_duplicate_mask = df.duplicated(
    subset=["Order ID"],
    keep=False
)
df[order_id_duplicate_mask].sort_values(by="Order ID")

,Order ID,Customer_ID,Order Date,Category,Product Name,Qty,Unit Price,Country,PaymentMethod,customer_email,Rating,Discount,Returned?
286,ORD1003,CUST274,09.05.2023,Books,wireless mouse,9,81.51,USA,COD,customer107@mail.com,1,,Yes
84,ORD1003,CUST297,04-06-2023,Electronics,Phone Case,8,60.5,France,Credit Card,customer253@mail.com,,0.06,1
414,ORD1006,CUST275,2023-01-27,ELECTRONICS,Smart Watch,10,76010.0,,COD,customer176@mail.com,4,0.08,True
98,ORD1006,CUST275,2023-01-27,ELECTRONICS,Smart Watch,10,76010.0,,COD,customer176@mail.com,4,0.08,True
307,ORD1009,CUST103,10-13-2024,Books,Running Shoes,2,146.34,U.S.A,Credit Card,customer75@mail.com,5,0.33,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,ORD1954,CUST24,29-Mar-2023,Clothing,Running Shoes,10,163.42,,PayPal,customer6@mail.com,4,0.49,Yes
603,ORD1968,CUST166,07/01/2023,Home & Kitchen,,9,$199.86,india,,customer234@mail.com,2,5%,Yes
687,ORD1968,CUST166,07/01/2023,Home & Kitchen,,9,$199.86,india,,customer234@mail.com,2,5%,Yes
966,ORD1971,CUST265,02.01.2024,Books,Laptop Stand,6,73.62,USA,Cash on Delivery,customer37@mail.com,3,0.02,No


In [15]:
# Check the unique values and their frequency in important categorical columns.

categorical_columns = [
    "Category",
    "Product Name",
    "Country",
    "PaymentMethod",
    "Returned?"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Category ---
Category
ELECTRONICS             90
Electronics             87
beauty                  81
Clothing                79
electronics             79
Home & Kitchen          76
Books                   75
Sports                  74
Beauty                  72
Home and Kitchen        71
Books                   69
clothing                58
  Home & Kitchen        13
  clothing              13
  Electronics           12
  Home and Kitchen      12
  electronics           12
  beauty                11
  Sports                10
  Books                  9
  Books                  8
  Clothing               7
  Beauty                 7
  ELECTRONICS            5
                         1
Name: count, dtype: int64

--- Product Name ---
Product Name
                         93
Notebook                 56
USB-C Cable              56
Denim Jacket             53
Headphones               52
T-Shirt                  52
Bluetooth Speaker        49
Face Cream               48
Shampoo      

In [16]:
# Display all unique values for each categorical column.

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].unique())


--- Category ---
['Clothing' 'Beauty' 'beauty ' 'ELECTRONICS' 'Electronics' 'clothing'
 '  Electronics  ' 'Books ' 'electronics' '  Clothing  '
 'Home and Kitchen' 'Books' 'Sports' '  Home & Kitchen  ' 'Home & Kitchen'
 '  Books  ' '  Books   ' '  Sports  ' '  Home and Kitchen  '
 '  clothing  ' '  beauty   ' '  ELECTRONICS  ' '' '  electronics  '
 '  Beauty  ']

--- Product Name ---
['Shampoo' '' 'T-Shirt' 'Bluetooth Speaker' 'Headphones' 'Yoga Mat'
 'wireless mouse' 'Smart Watch ' '  Power Bank  ' 'USB-C Cable'
 'Power Bank' 'Denim Jacket' ' Running Shoes' 'Wireless Mouse'
 '  Gel Pen  ' 'Notebook' 'Gel Pen' 'Face Cream' 'Laptop Stand'
 'Webcam HD' '  Face Cream  ' '   Running Shoes  ' 'Phone Case'
 '  Webcam HD  ' '  wireless mouse  ' '  T-Shirt  ' '  Wireless Mouse  '
 '  Headphones  ' '  Shampoo  ' '  Denim Jacket  ' '  Yoga Mat  '
 '  Laptop Stand  ' '  Bluetooth Speaker  ' '  Smart Watch   '
 '  USB-C Cable  ' '  Notebook  ' '  Phone Case  ']

--- Country ---
['Australia' 'Fran

In [17]:
# Count invalid emails containing '@@'.

invalid_email_count = df["customer_email"].str.contains(
    "@@",
    na=False
).sum()

print("Invalid emails containing '@@':", invalid_email_count)

Invalid emails containing '@@': 46


In [18]:
# Create a cross-frequency table between Category and Product Name.
# strip() removes extra spaces and lower() makes the comparison case-insensitive.

category_product_table = pd.crosstab(
    df["Category"].str.strip().str.lower(),
    df["Product Name"].str.strip().str.lower()
)

category_product_table

Product Name,,bluetooth speaker,denim jacket,face cream,gel pen,headphones,laptop stand,notebook,phone case,power bank,running shoes,shampoo,smart watch,t-shirt,usb-c cable,webcam hd,wireless mouse,yoga mat
Category,,,,,,,,,,,,,,,,,,
,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
beauty,21,12,8,14,10,7,6,8,5,10,3,6,11,10,7,11,15,7
books,12,7,8,13,9,7,7,8,8,11,8,7,6,7,8,9,19,7
clothing,10,8,8,5,6,8,7,8,7,8,11,7,9,9,15,8,11,12
electronics,24,17,18,9,16,20,15,18,20,11,14,19,12,11,16,12,22,11
home & kitchen,9,4,4,4,1,3,9,4,4,6,7,4,4,7,5,2,10,2
home and kitchen,12,3,5,3,2,8,2,5,4,5,0,3,6,8,1,2,9,5
sports,4,5,6,6,4,4,3,7,0,4,3,8,4,5,6,4,6,5


In [19]:
Numeric_Columns=['Qty', 'Unit Price', 'Rating', 'Discount']
for col in Numeric_Columns:
    df[col]=pd.to_numeric(df[col], errors='coerce')
    invalid_values=df[col][pd.to_numeric(df[col], errors='coerce').isna() & (df[col] != "")]
    print(f"\n{'=' * 50}")
    print(f"Non-convertible values in: {col}")
    print(f"{'=' * 50}")

    print(invalid_values)


Non-convertible values in: Qty
11     NaN
16     NaN
20     NaN
33     NaN
34     NaN
        ..
976    NaN
980    NaN
1001   NaN
1019   NaN
1028   NaN
Name: Qty, Length: 61, dtype: float64

Non-convertible values in: Unit Price
0      NaN
1      NaN
12     NaN
15     NaN
27     NaN
        ..
993    NaN
997    NaN
1004   NaN
1027   NaN
1030   NaN
Name: Unit Price, Length: 132, dtype: float64

Non-convertible values in: Rating
14    NaN
17    NaN
24    NaN
46    NaN
76    NaN
       ..
981   NaN
982   NaN
988   NaN
989   NaN
992   NaN
Name: Rating, Length: 86, dtype: float64

Non-convertible values in: Discount
0      NaN
3      NaN
4      NaN
7      NaN
9      NaN
        ..
1020   NaN
1022   NaN
1025   NaN
1029   NaN
1030   NaN
Name: Discount, Length: 419, dtype: float64


In [20]:
# --------------------------------------------------
# HINT 3: Check invalid_date string
# --------------------------------------------------

invalid_date_count = (
    df['Order Date']
    .astype(str)
    .str.strip()
    .eq('invalid_date')
    .sum()
)

print("Number of 'invalid_date' values:", invalid_date_count)

Number of 'invalid_date' values: 14


In [21]:
# --------------------------------------------------
# Try converting Order Date into datetime
# --------------------------------------------------

parsed_dates = pd.to_datetime(
    df['Order Date'].astype(str).str.strip(),
    errors='coerce'
)

print("Successfully parsed dates:", parsed_dates.notna().sum())
print("Failed to parse dates:", parsed_dates.isna().sum())

Successfully parsed dates: 156
Failed to parse dates: 875


In [22]:
# --------------------------------------------------
# Display invalid/unparseable dates
# --------------------------------------------------

invalid_dates = df[
    parsed_dates.isna()
]

print("Invalid / Unparseable dates:")
print(invalid_dates[['Order Date']])

Invalid / Unparseable dates:
       Order Date
1      30.06.2023
3     19-Sep-2024
4      2024-04-21
5      06-11-2023
7     16-May-2024
...           ...
1025   2024-09-16
1027   2023-07-22
1028   27.03.2024
1029   2023-06-28
1030  12-Mar-2024

[875 rows x 1 columns]


In [ ]:
import pandas as pd


numeric_cols = ['Qty', 'Unit Price', 'Rating', 'Discount']

for col in numeric_cols:
    converted = pd.to_numeric(df[col], errors='coerce')

    invalid_values = df[col][
        converted.isna() & (df[col] != "")
    ]

    print(f"\nNon-convertible values in '{col}':")
    print(invalid_values)




for col in numeric_cols:
    converted = pd.to_numeric(df[col], errors='coerce')

    print(f"\n--- {col} ---")
    print("Min:", converted.min())
    print("Max:", converted.max())



qty = pd.to_numeric(df['Qty'], errors='coerce')

print("\nNegative or Zero Quantity:")
print(df[qty <= 0])



unit_price = pd.to_numeric(df['Unit Price'], errors='coerce')

print("\nNegative Unit Price:")
print(df[unit_price < 0])



rating = pd.to_numeric(df['Rating'], errors='coerce')

print("\nRating greater than 5:")
print(df[rating > 5])



print("\nUnit Price greater than 10000:")
print(df[unit_price > 10000])



print("\nQuantity greater than 100:")
print(df[qty > 100])



invalid_date_count = (
    df['Order Date'].astype(str).str.strip() == 'invalid_date'
).sum()

print("\nCount of 'invalid_date':", invalid_date_count)



converted_dates = pd.to_datetime(
    df['Order Date'].str.strip(),
    errors='coerce'
)

print("\nParsed dates:")
print(converted_dates)



print("\nSuccessfully parsed dates:",
      converted_dates.notna().sum())


print("Failed to parse dates:",
      converted_dates.isna().sum())


Non-convertible values in 'Qty':
11     NaN
16     NaN
20     NaN
33     NaN
34     NaN
        ..
976    NaN
980    NaN
1001   NaN
1019   NaN
1028   NaN
Name: Qty, Length: 61, dtype: float64

Non-convertible values in 'Unit Price':
0      NaN
1      NaN
12     NaN
15     NaN
27     NaN
        ..
993    NaN
997    NaN
1004   NaN
1027   NaN
1030   NaN
Name: Unit Price, Length: 132, dtype: float64

Non-convertible values in 'Rating':
14    NaN
17    NaN
24    NaN
46    NaN
76    NaN
       ..
981   NaN
982   NaN
988   NaN
989   NaN
992   NaN
Name: Rating, Length: 86, dtype: float64

Non-convertible values in 'Discount':
0      NaN
3      NaN
4      NaN
7      NaN
9      NaN
        ..
1020   NaN
1022   NaN
1025   NaN
1029   NaN
1030   NaN
Name: Discount, Length: 419, dtype: float64

--- Qty ---
Min: -5.0
Max: 9999.0

--- Unit Price ---
Min: -160.93
Max: 187080.0

--- Rating ---
Min: 1.0
Max: 10.0

--- Discount ---
Min: 0.0
Max: 0.5

Negative or Zero Quantity:
     Order ID Customer_ID 